In [36]:
import requests

from typing import Any, overload, Mapping, List, Dict, Iterable, Optional
from pydantic import BaseModel, Field

import uuid
url = "http://10.42.0.1:8000/ingest/evidence"

from datetime import datetime, UTC
now = datetime.now(UTC).isoformat()
from datetime import datetime
from typing import Any, Literal

from pydantic import BaseModel, Field


class Evidence(BaseModel):
    evidence_id: str
    execution_id: str
    event_id: str

    evidence_type: str
    content: str
    content_hash: str | None = None

    source_type: str
    source_name: str | None = None
    source_url: str | None = None

    observed_at: datetime
    retrieved_at: datetime
    extraction_method: str | None = None

    parent_evidence_ids: list[str] = Field(default_factory=list)
    metadata: dict[str, Any] = Field(default_factory=dict)

    worker_version: str
    schema_version: Literal["evidence.v1"] = "evidence.v1"
    
class ToolEvent(BaseModel):
    iteration: int
    event_type: str
    tool: Optional[str] = None
    args: Dict[str, Any] = Field(default_factory=dict)
    result: Any = None


def ingest_tool_event(execution_id: str, tool_event: ToolEvent) -> None:
  
    try:
    
        now = str(datetime.now(UTC).isoformat())

        payload = Evidence(
            evidence_id=str(uuid.uuid4()),
            execution_id=execution_id,
            event_id=f"{execution_id}-{tool_event.iteration}",
            evidence_type=tool_event.tool,          # e.g. "web_search", "read_file"
            content=str(tool_event.result),         # jam for now, per your call
            content_hash="",
            source_type=tool_event.event_type,
            source_name=tool_event.tool,
            observed_at=now,
            retrieved_at=now,
            metadata={"args": tool_event.args},     # structured, don't lose it
            worker_version="1.0.1",
        )
        resp = requests.post(url, json=payload.model_dump(mode="json"), timeout=10)
        resp.raise_for_status()
    except Exception as e:
        # Log but do not raise – evidence is observational
        print(f"[Evidence ingestion] failed for event {tool_event}: {e}")




tool_result_event = ToolEvent(
        iteration=2,
        event_type="tool_call_result",
        tool='list_files',
        args={},
        result="""inference-backbone/\ninference-backbone/.gitignore\ninference-backbone/README.md\ninference-backbone/backbone-api/\ninference-backbone/backbone-api/Dockerfile\ninference-backbone/backbone-api/clients/\ninference-backbone/backbone-api/clients/__init__.py\ninference-backbone/backbone-api/clients/graphdb_client.py\ninference-backbone/backbone-api/clients/weaviate.py\ninference-backbone/backbone-api/main.py\ninference-backbone/backbone-api/projections/\ninference-backbone/backbone-api/projections/__init__.py\ninference-backbone/backbone-api/projections/graphdb.py\ninference-backbone/backbone-api/requirements.txt\ninference-backbone/backbone-api/routers/\ninference-backbone/backbone-api/routers/__init__.py\ninference-backbone/backbone-api/routers/health.py\ninference-backbone/backbone-api/routers/ingest.py\ninference-backbone/backbone-api/routers/retrival.py\ninference-backbone/backbone-api/routers/search.py\ninference-backbone/backbone-api/util/\ninference-backbone/backbone-api/util/chunker.py\ninference-backbone/backbone-api/util/embedding_provider.py\ninference-backbone/backbone-api/util/file_persistence.py\ninference-backbone/backbone-api/util/identity.py\ninference-backbone/contracts/\ninference-backbone/contracts/general_contracts/\ninference-backbone/contracts/general_contracts/comms.py\ninference-backbone/contracts/inference_contracts/\ninference-backbone/contracts/inference_contracts/__init__.py\ninference-backbone/contracts/inference_contracts/evidence.py\ninference-backbone/contracts/inference_contracts/evidence_chunk.py\ninference-backbone/docker-compose.yml\ninference-backbone/repository.init/\ninference-backbone/repository.init/inference-backbone/\ninference-backbone/repository.init/inference-backbone/config.ttl\ninference-backbone/repository.init/inference-backbone/schema.ttl\ninference-backbone/validator-agent/\ninference-backbone/validator-agent/Dockerfile\ninference-backbone/validator-agent/__init__.py\ninference-backbone/validator-agent/main.py\ninference-backbone/validator-agent/requirements.txt\ninference-backbone/validator-agent/test_semantic_memory_integration.py\ninference-backbone/worker-agent/\ninference-backbone/worker-agent/Dockerfile\ninference-backbone/worker-agent/main.py\ninference-backbone/worker-agent/requirements.txt""")

ingest_tool_event(str(uuid.uuid4()), tool_result_event)


In [35]:
import requests

url = "http://10.42.0.1:8000/ingest/evidence"

from datetime import datetime, UTC
now = datetime.now(UTC).isoformat()

payload = {
   "evidence_id": "ev-003",
  "execution_id": "exec-42",
  "event_id": "evt-8",
  "evidence_type": "web_search_result",
  "content": "search result snippet text",
  "content_hash": "sha256:def456...",
  "source_type": "web_search",
  "source_name": "duckduckgo",
  "source_url": "https://example.com/article",
  "observed_at": now,
  "retrieved_at": now,
  "extraction_method": "html_to_text",
  "parent_evidence_ids": ["ev-001"],
  "metadata": {'path': 'inference-backbone/backbone-api/routers/ingest.py', 'line_start': 1, 'line_end': 400} ,
  "worker_version": "0.1.0"
}

try:
    response = requests.post(
        url,
        json=payload,  # Serializes the dict and sets Content-Type: application/json
        timeout=1000,
    )

    response.raise_for_status()

    # If the endpoint returns JSON:
    result = response.json()

except requests.exceptions.RequestException as error:
    print(f"HTTP request failed: {error}")

result


HTTP request failed: 500 Server Error: Internal Server Error for url: http://10.42.0.1:8000/ingest/evidence


{'parent_id': '1ec0b59c-03ab-58a5-9cb4-d73ba59b9c8d',
 'chunk_count': 1,
 'chunks': ['518ee5b8-7940-4f72-a6df-7e359d45aedd']}

In [4]:
import requests

url = "http://10.42.0.1:8000/search/evidence"

payload = {
  "query": "list_files",
  "limit" : 500
}

try:
    response = requests.post(
        url,
        json=payload,  # Serializes the dict and sets Content-Type: application/json
        timeout=1000,
    )

    response.raise_for_status()

    # If the endpoint returns JSON:
    result = response.json()

except requests.exceptions.RequestException as error:
    print(f"HTTP request failed: {error}")

for r in result:
    print(r['distance'])


0.48718690872192383
0.5729252099990845
0.5886058211326599
0.6125339865684509
0.6125339865684509
